# Image Preprocessing for SpendWise Receipt OCR Pipeline

In the **SpendWise** project, we aim to accurately extract information like total amount, date, and store name from receipt photos using OCR. However, raw images captured in real-world conditions often contain noise, varying lighting, low contrast, and faded text — all of which significantly reduce OCR accuracy.

This notebook (`02_preprocessing.ipynb`) implements a preprocessing pipeline using OpenCV to clean and enhance receipt images before feeding them into OCR engines. Proper preprocessing is one of the most important steps in any document analysis pipeline, as it can dramatically improve text recognition performance.

## 1. Importing Required Libraries

This cell imports the libraries needed for image processing and data handling.

In [1]:
import cv2
import numpy as np
from pathlib import Path
import pandas as pd

**Why these libraries?**
- **OpenCV (cv2)**: Industry standard for computer vision tasks like denoising, contrast enhancement, and thresholding.
- **NumPy**: Supports array operations used internally by OpenCV.
- **Pathlib & Pandas**: Consistent file handling and loading of our ground truth metadata.

This setup provides all the tools needed for efficient batch image preprocessing.

## 2. Setting Up Directory Paths

This cell defines the input (raw) and output (processed) directories for our images.

In [2]:
raw_dir = Path("../data/raw")
processed_dir = Path("../data/processed")
processed_dir.mkdir(exist_ok=True)

We separate raw and processed images to preserve the original data while creating an optimized version for OCR. Using `mkdir(exist_ok=True)` ensures the directory is created only if it doesn't already exist, making the notebook safe to re-run.

## 3. Defining the Preprocessing Function

This cell defines the core `preprocess_image()` function that applies a sequence of enhancements to each receipt.

In [3]:
def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=8)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(denoised)
    
    binary = cv2.adaptiveThreshold(enhanced, 255, 
                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY, 31, 10)
    return binary

**Step-by-step explanation and justification:**

1. **Grayscale conversion** (`COLOR_BGR2GRAY`): Removes color information we don’t need for text extraction, reducing computational load and focusing on intensity.

2. **fastNlMeansDenoising (h=8)**: Removes sensor noise and minor artifacts common in phone-captured images. The `h=8` value offers a good balance — strong enough to clean noise but gentle enough to preserve fine text.

3. **CLAHE** (Contrast Limited Adaptive Histogram Equalization): Works locally on small tiles and limits over-amplification. Excellent for receipts with both fresh and faded areas.

4. **Adaptive Gaussian Thresholding**: Handles uneven lighting much better than global thresholding.

**Important**: All steps after grayscale conversion operate on single-channel images, which is why `BGR2GRAY` is required.

## 4. Applying Preprocessing to All Receipts

This cell loads the ground truth and processes every receipt image, saving the enhanced versions.

In [4]:
df = pd.read_csv("../data/annotated/ground_truth.csv")

for _, row in df.iterrows():
    img = cv2.imread(str(raw_dir / row['filename']))
    if img is not None:
        processed = preprocess_image(img)
        cv2.imwrite(str(processed_dir / row['filename']), processed)
        print(f"✅ Preprocessed: {row['filename']}")

✅ Preprocessed: receipt_01.jpg
✅ Preprocessed: receipt_02.jpg
✅ Preprocessed: receipt_03.jpg
✅ Preprocessed: receipt_04.jpg
✅ Preprocessed: receipt_05.jpg
✅ Preprocessed: receipt_06.jpg
✅ Preprocessed: receipt_07.jpg
✅ Preprocessed: receipt_08.jpg
✅ Preprocessed: receipt_09.jpg
✅ Preprocessed: receipt_10.jpg
✅ Preprocessed: receipt_11.jpg
✅ Preprocessed: receipt_12.jpg
✅ Preprocessed: receipt_13.jpg
✅ Preprocessed: receipt_14.jpg
✅ Preprocessed: receipt_15.jpg
✅ Preprocessed: receipt_16.jpg
✅ Preprocessed: receipt_17.jpg
✅ Preprocessed: receipt_18.jpg
✅ Preprocessed: receipt_19.jpg
✅ Preprocessed: receipt_20.jpg
✅ Preprocessed: receipt_21.jpg
✅ Preprocessed: receipt_22.jpg


This batch processing loop ensures every image in our dataset receives consistent preprocessing. The success messages confirm all 22 receipts were processed successfully. The processed images are now ready for OCR testing and model development.

## Summary of Preprocessing Pipeline

This notebook successfully created a **cleaned and enhanced version** of all 22 receipt images in the `../data/processed/` directory.

**Key benefits of our pipeline:**
- Reduced noise while preserving text edges
- Improved contrast in both fresh and faded receipts
- Better handling of uneven lighting through adaptive methods
- Produced clean binary images optimized for OCR engines like Tesseract

The techniques chosen (especially CLAHE + adaptive thresholding) are well-suited for document images and should significantly boost OCR accuracy compared to using raw photos. 

**Next steps**: Test OCR performance on both raw and processed images, then experiment with hyperparameter tuning or additional techniques like deskewing and morphological operations.